In [0]:
from pyspark.sql.functions import explode_outer, col
from pyspark.sql.types import StructType, ArrayType

def auto_flatten(df):

    while True:
        complex_col = None

        for field in df.schema.fields:

            if isinstance(field.dataType, StructType):
                complex_col = field.name

                expanded_cols = [
                    col(f"{complex_col}.{nested.name}").alias(f"{complex_col}_{nested.name}")
                    for nested in field.dataType.fields
                ]

                df = df.select("*", *expanded_cols).drop(complex_col)
                break

            elif isinstance(field.dataType, ArrayType):
                complex_col = field.name
                df = df.withColumn(
                    complex_col,
                    explode_outer(col(complex_col))
                )
                break

        if complex_col is None:
            break

    return df